# 01 — Supply Chain Network Optimization

In [2]:
%cd /content
!test -d supply-chain-network-optimization || git clone https://github.com/hemanthkarthikd/supply-chain-network-optimization.git
%cd /content/supply-chain-network-optimization/notebooks

/content
/content/supply-chain-network-optimization/notebooks


In [3]:
import pandas as pd
import numpy as np
from scipy.optimize import milp, LinearConstraint, Bounds

plants = pd.read_csv("../data/raw/plants.csv")
dcs = pd.read_csv("../data/raw/distribution_centers.csv")
customers = pd.read_csv("../data/raw/customer_zones.csv")
pdc = pd.read_csv("../data/raw/plant_to_dc_costs.csv")
dcc = pd.read_csv("../data/raw/dc_to_customer_costs.csv")

print("Total plant capacity:", plants["capacity_units"].sum())
print("Total customer demand:", customers["demand_units"].sum())

Total plant capacity: 180000
Total customer demand: 149500


In [4]:
P = list(plants["plant_id"])
D = list(dcs["dc_id"])
C = list(customers["customer_id"])

x_keys = [(p,d) for p in P for d in D]      # plant -> DC
z_keys = [(d,c) for d in D for c in C]      # DC -> customer
y_keys = D                                  # open/close DC

var_keys = [("x",)+k for k in x_keys] + [("z",)+k for k in z_keys] + [("y",d) for d in y_keys]
idx = {k:i for i,k in enumerate(var_keys)}
n = len(var_keys)

obj = np.zeros(n)

pdc_cost = {(r.plant_id,r.dc_id): r.transport_cost_per_unit for _,r in pdc.iterrows()}
dcc_cost = {(r.dc_id,r.customer_id): r.transport_cost_per_unit for _,r in dcc.iterrows()}
fixed_cost = dict(zip(dcs["dc_id"], dcs["fixed_annual_cost"]))
prod_cost = dict(zip(plants["plant_id"], plants["production_cost_per_unit"]))

for p,d in x_keys:
    obj[idx[("x",p,d)]] = prod_cost[p] + pdc_cost[(p,d)]
for d,cust in z_keys:
    obj[idx[("z",d,cust)]] = dcc_cost[(d,cust)]
for d in D:
    obj[idx[("y",d)]] = fixed_cost[d]

A, lb, ub = [], [], []

plant_cap = dict(zip(plants["plant_id"], plants["capacity_units"]))
for p in P:
    row = np.zeros(n)
    for d in D:
        row[idx[("x",p,d)]] = 1
    A.append(row); lb.append(-np.inf); ub.append(plant_cap[p])

# DC flow balance
for d in D:
    row = np.zeros(n)
    for p in P:
        row[idx[("x",p,d)]] = 1
    for cust in C:
        row[idx[("z",d,cust)]] -= 1
    A.append(row); lb.append(0); ub.append(0)

# DC capacity tied to open decision
dc_cap = dict(zip(dcs["dc_id"], dcs["capacity_units"]))
for d in D:
    row = np.zeros(n)
    for cust in C:
        row[idx[("z",d,cust)]] = 1
    row[idx[("y",d)]] = -dc_cap[d]
    A.append(row); lb.append(-np.inf); ub.append(0)

# Customer demand
dem = dict(zip(customers["customer_id"], customers["demand_units"]))
for cust in C:
    row = np.zeros(n)
    for d in D:
        row[idx[("z",d,cust)]] = 1
    A.append(row); lb.append(dem[cust]); ub.append(dem[cust])

constraints = LinearConstraint(np.array(A), np.array(lb), np.array(ub))

lower = np.zeros(n)
upper = np.full(n, np.inf)
integrality = np.zeros(n, dtype=int)

for d in D:
    upper[idx[("y",d)]] = 1
    integrality[idx[("y",d)]] = 1

bounds = Bounds(lower, upper)

res = milp(c=obj, constraints=constraints, bounds=bounds, integrality=integrality)

print("Optimization success:", res.success)
print("Minimum annual network cost:", round(res.fun,2))

Optimization success: True
Minimum annual network cost: 10948295.0


In [5]:
solution = res.x

open_dcs = [d for d in D if solution[idx[("y",d)]] > 0.5]
print("Open DCs:", open_dcs)

plant_dc_flows = []
for p,d in x_keys:
    v = solution[idx[("x",p,d)]]
    if v > 1e-6:
        plant_dc_flows.append([p,d,v])

dc_customer_flows = []
for d,cust in z_keys:
    v = solution[idx[("z",d,cust)]]
    if v > 1e-6:
        dc_customer_flows.append([d,cust,v])

plant_dc_flows = pd.DataFrame(plant_dc_flows, columns=["plant_id","dc_id","flow_units"])
dc_customer_flows = pd.DataFrame(dc_customer_flows, columns=["dc_id","customer_id","flow_units"])
open_dc_table = dcs[dcs["dc_id"].isin(open_dcs)].copy()

plant_dc_flows.to_csv("../data/processed/optimized_plant_dc_flows.csv", index=False)
dc_customer_flows.to_csv("../data/processed/optimized_dc_customer_flows.csv", index=False)
open_dc_table.to_csv("../data/processed/open_distribution_centers.csv", index=False)

display(open_dc_table[["dc_id","dc_name","capacity_units","fixed_annual_cost"]])
display(plant_dc_flows)
display(dc_customer_flows.head(30))

Open DCs: ['D1', 'D3', 'D4', 'D5']


,dc_id,dc_name,capacity_units,fixed_annual_cost
0,D1,Boston DC,42000,1800000
2,D3,Dallas DC,52000,1900000
3,D4,Atlanta DC,45000,1750000
4,D5,Denver DC,38000,1650000


,plant_id,dc_id,flow_units
0,P1,D1,34000.0
1,P1,D5,10500.0
2,P2,D3,51000.0
3,P2,D5,9000.0
4,P3,D4,45000.0


,dc_id,customer_id,flow_units
0,D1,C1,12000.0
1,D1,C2,14500.0
2,D1,C3,7500.0
3,D3,C7,16000.0
4,D3,C8,9500.0
5,D3,C10,10000.0
6,D3,C11,15500.0
7,D4,C3,5500.0
8,D4,C4,15000.0
9,D4,C5,13500.0
